# 3. Train Config

Tao file YAML cau hinh cho ai-toolkit.

In [ ]:
import os
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")
print(f"Current working directory: {Path.cwd()}")

In [ ]:
import sys
sys.path.append('scripts')
from config_generator import generate_config
import os
from pathlib import Path

project_name = "db9_toolkit_trainner"
model_path = "black-forest-labs/FLUX.2-klein-base-9B"

base_dir = Path.cwd()
dataset_path = str(base_dir / "datasets" / "processed").replace("\", "/")
output_path = str(base_dir / "outputs").replace("\", "/")

lora_rank = 64
lora_alpha = 64
batch_size = 6
learning_rate = 4e-4
train_steps = 2000
gradient_accumulation = 1
resolution = 1536
enable_bucketing = True
bucket_step = 64
min_bucket_reso = 512
max_bucket_reso = 2048
flip_aug = False
color_aug = False
optimizer = "adamw8bit"
lr_scheduler = "constant_with_warmup"
warmup_steps = 100
gradient_checkpointing = True
quantize = False
save_every = 500
sample_every = 500
sample_prompts = [
    "A portrait of a person in DB9 style, highly detailed"
]

config_yaml = generate_config(
    project_name=project_name,
    model_path=model_path,
    dataset_path=dataset_path,
    output_path=output_path,
    lora_rank=lora_rank,
    lora_alpha=lora_alpha,
    batch_size=batch_size,
    learning_rate=learning_rate,
    train_steps=train_steps,
    gradient_accumulation=gradient_accumulation,
    resolution=resolution,
    enable_bucketing=enable_bucketing,
    bucket_step=bucket_step,
    min_bucket_reso=min_bucket_reso,
    max_bucket_reso=max_bucket_reso,
    flip_aug=flip_aug,
    color_aug=color_aug,
    optimizer=optimizer,
    lr_scheduler=lr_scheduler,
    warmup_steps=warmup_steps,
    gradient_checkpointing=gradient_checkpointing,
    quantize=quantize,
    save_every=save_every,
    sample_every=sample_every,
    sample_prompts=sample_prompts,
)

os.makedirs("configs", exist_ok=True)
config_path = f"configs/{project_name}.yaml"
with open(config_path, "w", encoding="utf-8") as f:
    f.write(config_yaml)

print(f"Config saved to {config_path}")

## Flux 2 Klein Validation

In [ ]:
FLUX2_KLEIN_MAX_RESOLUTION = 2048

print("Flux 2 Klein validation")
print(f"- Max supported resolution: {FLUX2_KLEIN_MAX_RESOLUTION}px")
print(f"- Current resolution: {resolution}px")
print(f"- Bucket range: {min_bucket_reso}-{max_bucket_reso}px, step {bucket_step}px")

if resolution > FLUX2_KLEIN_MAX_RESOLUTION:
    raise ValueError("Flux 2 Klein max resolution is 2048px. Lower resolution before generating config.")
if max_bucket_reso > FLUX2_KLEIN_MAX_RESOLUTION:
    raise ValueError("Flux 2 Klein max bucket resolution is 2048px. Lower max_bucket_reso before generating config.")
if min_bucket_reso > max_bucket_reso:
    raise ValueError("min_bucket_reso cannot be greater than max_bucket_reso.")
if bucket_step <= 0 or min_bucket_reso % bucket_step != 0 or max_bucket_reso % bucket_step != 0:
    raise ValueError("Bucket min/max must be divisible by a positive bucket_step.")
if enable_bucketing and not (min_bucket_reso <= resolution <= max_bucket_reso):
    raise ValueError("Resolution must be inside the bucket range when bucketing is enabled.")

print("Validation passed.")